# Whisper fine-tuning on a Colab GPU

This notebook is a thin wrapper around the same `whisper/scripts/finetune.py` used for local CPU smoke tests - no script changes, just a GPU environment and GPU-scale hyperparameters (from `whisper/lib/default_config.yaml`) instead of a language's CPU overrides.

**Before running:**
- In Colab, go to `Runtime > Change runtime type` and select a GPU (T4 on the free tier).
- `ushahidi/dv-transcription-analysis` is a **private** repo, so cloning it here needs a GitHub Personal Access Token: create one at [github.com/settings/tokens](https://github.com/settings/tokens) (fine-grained, scoped to just this repo, with read-only "Contents" access is enough). The next cell prompts for it with `getpass` so it's typed once per session and never saved to disk or to this notebook file.

See `whisper/README.md` for the full pipeline explanation.

In [ ]:
from getpass import getpass

github_token = getpass("GitHub personal access token (read access to ushahidi/dv-transcription-analysis): ")
!git clone --branch main https://{github_token}@github.com/ushahidi/dv-transcription-analysis.git
%cd dv-transcription-analysis
del github_token  # don't keep it around in memory longer than needed

In [ ]:
# Colab ships a CUDA-enabled torch already - only the project's other deps are needed.
!pip install -q -r requirements.txt

## Pull the dataset

Increase `--max-samples` well beyond the local CPU smoke-test size now that a GPU can handle real training data volumes.

In [ ]:
LANGUAGE = "kiswahili"  # or "english"

!python whisper/scripts/prepare_dataset.py --language {LANGUAGE} --split train --max-samples 2000
!python whisper/scripts/prepare_dataset.py --language {LANGUAGE} --split validation --max-samples 200
!python data/scripts/prepare_dataset.py --language {LANGUAGE} --split test --max-samples 200

## Fine-tune with GPU-scale hyperparameters

Pointing `--config` at `whisper/lib/default_config.yaml` skips the language's CPU-smoke-test overrides (small `max_steps`, tiny batch size) and uses the full recipe from the Hugging Face fine-tuning blog instead (`lr=1e-5`, `warmup_steps=500`, `max_steps=5000`, `batch_size=16`, `fp16=True`).

`--model` can be bumped up from `whisper-tiny`/`whisper-base` to `whisper-small` or larger now that a real GPU is available.

In [ ]:
!python whisper/scripts/finetune.py \
  --language {LANGUAGE} \
  --model openai/whisper-small \
  --config whisper/lib/default_config.yaml

If the runtime disconnects mid-training, re-run the cell above with `--resume-from-checkpoint whisper/<language>/models/<checkpoint-dir>/checkpoint-<step>` to continue from the last saved checkpoint (training args save every `save_steps`, so at most that many steps are lost).

## Evaluate and download the checkpoint

Zip the fine-tuned model directory and download it so it can be brought back to the local repo for `export_model.py`, or export directly to CTranslate2 here on the GPU box first.

In [ ]:
!python whisper/scripts/run_baseline.py --language {LANGUAGE} --model whisper/{LANGUAGE}/models/whisper-small-finetuned --split test

In [ ]:
import shutil
from google.colab import files

model_dir = f"whisper/{LANGUAGE}/models/whisper-small-finetuned"
archive_path = shutil.make_archive("finetuned-model", "zip", model_dir)
files.download(archive_path)